# Baselines

Edit **DATASET** and **ENCODER** in the code cell below, then run all.

In [1]:
import sys
from pathlib import Path
import os
import time
import json
import hashlib

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "modules").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.preprocessing import preprocess
from utils.utils import dataset_table_path,save_results
from modules.encoders import get_encoder
from modules.clustering import get_clusterer
from evaluation.evaluator import Evaluator
import random

import numpy as np
import pandas as pd
import yaml

In [2]:
def estimate_k(embeddings, k_min=2, k_max=20, seed=42):
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

    best_k = k_min
    best_score = -1

    for k in range(k_min, min(k_max, len(embeddings))):
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels = km.fit_predict(embeddings)

        if len(set(labels)) < 2:
            continue

        score = silhouette_score(embeddings, labels)

        if score > best_score:
            best_score = score
            best_k = k

    return best_k

In [3]:
def run_baseline(dataset_name, encoder_name, clusterer_name, cfg):
    cfg["dataset"]["name"] = dataset_name
    cfg["encoder"]["name"] = encoder_name
    cfg["clustering"]["name"] = clusterer_name

    path = dataset_table_path(REPO_ROOT, dataset_name)
    df = pd.read_csv(path)

    text_col = cfg["dataset"]["text_col"]
    label_col = cfg["dataset"]["label_col"]
    cap = cfg["dataset"].get("max_samples")

    
    df = df.dropna(subset=[text_col, label_col])
    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 0]

    if cap is not None and cap < len(df):
        df = df.sample(n=cap, random_state=cfg["experiment"]["seed"])

    texts = df[text_col].tolist()
    y_true = df[label_col].astype(int).to_numpy()

    seed = cfg["experiment"]["seed"]
    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except:
        pass

    
    emb_hash_input = {
        "encoder": cfg["encoder"],
        "preprocessing": cfg["preprocessing"],
        "dataset": dataset_name,
        "cap": cap
    }

    emb_cfg_hash = hashlib.md5(
        json.dumps(emb_hash_input, sort_keys=True).encode()
    ).hexdigest()[:8]

    embeddings_path = Path(
        f"../artifacts/embeddings/emb_{dataset_name}_{encoder_name}_{emb_cfg_hash}.npy"
    )
    embeddings_path.parent.mkdir(parents=True, exist_ok=True)

    t0 = time.time()

    if not embeddings_path.exists():
        print("Generating embeddings")

        if encoder_name in cfg["preprocessing"]["enabled_for"]:
            cleaned, tokenized = preprocess(texts)
        else:
            cleaned, tokenized = texts, None

        encoder = get_encoder(cfg['encoder'])
        embeddings = encoder.fit_transform(cleaned, tokenized_texts=tokenized)

        np.save(embeddings_path, embeddings)
    else:
        print("Embeddings loaded (cached)")
        embeddings = np.load(embeddings_path)

    embedding_time = time.time() - t0

    reduction_cfg = cfg.get("reduction", {})
    if reduction_cfg.get("method") == "pca":
        from sklearn.decomposition import PCA
        embeddings = PCA(
            n_components=reduction_cfg.get("n_components", 0.95),
            random_state=seed
        ).fit_transform(embeddings)

    
    true_k = len(np.unique(y_true))
    k_cfg = cfg["clustering"].get("k", "true")

    if k_cfg == "true":
        k = true_k
    elif k_cfg == "auto":
        print("Estimating k automatically...")
        k = estimate_k(
            embeddings,
            k_min=2,
            k_max=cfg["clustering"].get("k_max", 20),
            seed=seed
        )
        print(f"Estimated k: {k}")
    else:
        k = int(k_cfg)

    n_runs = cfg["experiment"].get("n_runs",1)

    eval_mode = cfg["evaluation"].get("mode", "both")

    all_metrics = []
    all_times = []

    for run in range(n_runs):
        run_seed = seed + run
        np.random.seed(run_seed)
        random.seed(run_seed)

        t1 = time.time()
        clusterer = get_clusterer(cfg["clustering"])
        y_pred = clusterer.fit_predict(embeddings, k)
        clustering_time = time.time() - t1

        unique_clusters = len(np.unique(y_pred))
        collapse_flag = unique_clusters < k

        metrics = {}

        if eval_mode in ["label", "both"]:
            evaluator = Evaluator(
                metrics=cfg["evaluation"]["metrics"]["label"])
            metrics.update(evaluator.evaluate(y_true, y_pred))

        if eval_mode in ["intrinsic", "both"]:
            from sklearn.metrics import silhouette_score, davies_bouldin_score
            try:
                metrics["silhouette"] = silhouette_score(embeddings, y_pred)
                metrics["davies_bouldin"] = davies_bouldin_score(
                    embeddings, y_pred)
            except:
                metrics["silhouette"] = None
                metrics["davies_bouldin"] = None

        metrics["collapse"] = collapse_flag

        all_metrics.append(metrics)
        all_times.append(clustering_time)


    def aggregate(metric_list):
        keys = metric_list[0].keys()
        agg = {}
        for k in keys:
            values = [m[k] for m in metric_list if m[k] is not None]
            if len(values) > 0 and isinstance(values[0], (int, float, np.floating)):
                agg[f"{k}_mean"] = float(np.mean(values))
                agg[f"{k}_std"] = float(np.std(values))
            else:
                agg[k] = values[0] if values else None
        return agg

    final_metrics = aggregate(all_metrics)

    results = {
        # Experiment metadata
        "experiment_name": cfg["experiment"]["name"],
        "dataset": dataset_name,
        "encoder": encoder_name,
        "clusterer": clusterer_name,

        # Data
        "n_docs": len(texts),
        "k_used": k,
        "k_true": true_k,

        # Timing
        "embedding_time": embedding_time,
        "clustering_time_mean": float(np.mean(all_times)),
        "clustering_time_std": float(np.std(all_times)),
        "total_time": embedding_time + float(np.mean(all_times)),

        # Label metrics
        "acc_mean": final_metrics.get("acc_mean"),
        "acc_std": final_metrics.get("acc_std"),
        "nmi_mean": final_metrics.get("nmi_mean"),
        "nmi_std": final_metrics.get("nmi_std"),
        "ari_mean": final_metrics.get("ari_mean"),
        "ari_std": final_metrics.get("ari_std"),
        "purity_mean": final_metrics.get("purity_mean"),
        "purity_std": final_metrics.get("purity_std"),

        # Intrinsic metrics
        "silhouette_mean": final_metrics.get("silhouette_mean"),
        "silhouette_std": final_metrics.get("silhouette_std"),
        "davies_bouldin_mean": final_metrics.get("davies_bouldin_mean"),
        "davies_bouldin_std": final_metrics.get("davies_bouldin_std"),

        # Stability
        "collapse_mean": final_metrics.get("collapse_mean"),
    }

    save_results(results)

    return results

In [4]:
with open(REPO_ROOT / "config" / "default.yaml") as f:
    cfg = yaml.safe_load(f)
#'bbc_news','reuters','20newsgroups','agnews',
for dataset in ['dbpedia']:
    for encoder in ['tfidf','doc2vec','sbert']:
        for clusterer in ['kmeans','dec','ae']:
            print(f"\nRunning baseline for {dataset}-{encoder}-{clusterer}")
            run_baseline(dataset,encoder,clusterer,cfg)


Running baseline for dbpedia-tfidf-kmeans
Embeddings loaded (cached)
Estimating k automatically...


KeyboardInterrupt: 